## Business Question

Vanguard introduced a redesigned digital interface to improve the client experience during an online process.  
An A/B test was conducted to evaluate its effectiveness:

- **Control Group (A):** Existing interface  
- **Test Group (B):** Redesigned interface  

The goal of this analysis is to determine whether the new interface improves **process completion rate and user engagement**, and whether it achieves at least a **5% improvement in completion rate**, the minimum required to justify implementation.

## Metrics Analyzed

### 1. Completion Rate
Percentage of visits that reach the final step (`confirm`).

Completion Rate = Completed Visits / Total Visits  

This is the **primary business metric**.

### 2. Error Rate
Percentage of visits where users move backward to a previous step during the process.

Error Rate = Visits with Errors / Total Visits  

Backward navigation may indicate confusion or difficulty in the process.

### 3. Time Spent per Journey
Measures how long users take to move between steps during a visit.

This metric helps evaluate the **efficiency of the user journey** and whether the redesign affects user behavior.

### Load and Sort the Data

First, we load the cleaned dataset and sort the dataframe by datetime for each visit to reconstruct the user journey.  
We then verify that each `visit_id` is associated with only one variation (either **Test** or **Control**, but not both).

In [1]:
import pandas as pd
import scipy.stats as st
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

In [2]:
df_web = pd.read_csv('/Users/hanguyen/Documents/DATA ANALYTICS/IRONHACK/ab-testing-vanguard-digital-journey/data/df_web.csv')

In [3]:
df_web.sort_values(by=["client_id", "visit_id" ,"date_time"], inplace=True)
df_web

,client_id,visitor_id,visit_id,process_step,date_time,Variation
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
...,...,...,...,...,...,...
314459,9999729,834634258_21862004160,870243567_56915814033_814203,confirm,2017-05-08 16:09:40,Test
314460,9999729,604429154_69247391147,99583652_41711450505_426179,start,2017-04-05 13:40:49,Test
314461,9999729,604429154_69247391147,99583652_41711450505_426179,step_1,2017-04-05 13:41:04,Test
314462,9999832,145538019_54444341400,472154369_16714624241_585315,start,2017-05-16 16:46:03,Test


In [ ]:
### Checking if each visit_id is mapped to a single Variation (either Test or Control, never both)

df_web.groupby("visit_id")["Variation"].nunique().reset_index().sort_values(by="Variation", ascending=False)

,visit_id,Variation
0,100012776_37918976071_457913,1
45981,700355178_31623928130_462227,1
45967,700190990_11171081628_520439,1
45968,70020077_40688951654_976127,1
45969,700211866_9445656730_286696,1
...,...,...
22992,402170104_12902080536_238460,1
22993,402186373_60129242148_182347,1
22994,402203822_54194913691_872242,1
22995,402234626_19272441400_686123,1


## 1. Completion Rate

Completion rate was analyzed to determine whether the redesigned interface improved the likelihood that users complete the process.

A visit was classified as completed if the maximum process step reached was `confirm`. Completion rates were then calculated separately for the Control and Test groups.

Results show that the Test group achieved a higher completion rate than the Control group.

In [4]:
### Completion Rate

#---> #visit completed/#total_visits

df_web["process_step_encoded"] = df_web["process_step"].map({"start":0,
                                                             "step_1":1,
                                                             "step_2":2,
                                                             "step_3":3,
                                                             "confirm":4})

df_web

,client_id,visitor_id,visit_id,process_step,date_time,Variation,process_step_encoded
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,0
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,1
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,2
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,3
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,4
...,...,...,...,...,...,...,...
314459,9999729,834634258_21862004160,870243567_56915814033_814203,confirm,2017-05-08 16:09:40,Test,4
314460,9999729,604429154_69247391147,99583652_41711450505_426179,start,2017-04-05 13:40:49,Test,0
314461,9999729,604429154_69247391147,99583652_41711450505_426179,step_1,2017-04-05 13:41:04,Test,1
314462,9999832,145538019_54444341400,472154369_16714624241_585315,start,2017-05-16 16:46:03,Test,0


In [5]:
df_cr = df_web.groupby(["Variation","visit_id"])["process_step_encoded"].max().reset_index()
df_cr["completed"] = df_cr["process_step_encoded"] == 4
df_cr


,Variation,visit_id,process_step_encoded,completed
0,Control,100030127_47967100085_936361,0,False
1,Control,100037962_47432393712_705583,1,False
2,Control,100057941_88477660212_944512,3,False
3,Control,10006594_66157970412_679648,4,True
4,Control,100071743_53464757454_616703,0,False
...,...,...,...,...
68958,Test,999960019_60838685252_926860,2,False
68959,Test,999971096_28827267783_236076,4,True
68960,Test,999976049_95772503197_182554,4,True
68961,Test,999984454_18731538378_781808,4,True


In [6]:
crosstab_var_c = pd.crosstab(df_cr["Variation"], df_cr["completed"])

In [7]:
crosstab_var_c

completed,False,True
Variation,,
Control,16123,15892
Test,15390,21558


In [55]:
#H0: p_Confirm Test - p_Confirm Control >= 5%
#H1: p_Confirm Test - p_Confirm Control < 5%

#sl=0.05

count = np.array([21558, 15892])

nobs = np.array([(15390+21558), (16123+15892)])

proportions_ztest(count, nobs, value=0.05, alternative="smaller")



(np.float64(9.74779514504426), np.float64(1.0))

In [8]:
print("Completion Rate for Test: ",21558/(15390+21558))
print("Completion Rate for Control: ", 15892/(16123+15892))

Completion Rate for Test:  0.5834686586554076
Completion Rate for Control:  0.49639231610182727


### Hypothesis Testing Result

To evaluate whether the improvement meets Vanguard’s business requirement, we tested the following hypotheses:

- **H₀:** p(Test) − p(Control) ≥ 5%  
- **H₁:** p(Test) − p(Control) < 5%

Significance level: α = 0.05

The z-test produced:

- **z-statistic:** 9.75  
- **p-value:** 1.00  

Since the p-value is much greater than 0.05, we **fail to reject the null hypothesis**.

### Interpretation

The results indicate that the improvement in completion rate is **not statistically smaller than the required 5% threshold**. Therefore, the redesigned interface **meets Vanguard’s business requirement** and shows a meaningful improvement in process completion.

## 2. Error Rate Analysis

Error rate measures the share of visits in which users move backward from a later step to an earlier step during the same visit.

Error Rate = Visits with Errors / Total Visits

Backward navigation may indicate confusion, hesitation, or difficulty understanding the process flow.

In [9]:
df_web["step_diff"] = df_web.groupby("visit_id")["process_step_encoded"].diff()

In [10]:
df_error = df_web.groupby(["Variation","visit_id"])["step_diff"].min().reset_index()
df_error["is_error"] = df_error["step_diff"] < 0
df_error

,Variation,visit_id,step_diff,is_error
0,Control,100030127_47967100085_936361,NaN,False
1,Control,100037962_47432393712_705583,-1.0,True
2,Control,100057941_88477660212_944512,-2.0,True
3,Control,10006594_66157970412_679648,-1.0,True
4,Control,100071743_53464757454_616703,NaN,False
...,...,...,...,...
68958,Test,999960019_60838685252_926860,-2.0,True
68959,Test,999971096_28827267783_236076,0.0,False
68960,Test,999976049_95772503197_182554,0.0,False
68961,Test,999984454_18731538378_781808,1.0,False


In [17]:
crosstab = pd.crosstab(df_error["Variation"], df_error["is_error"])
crosstab

is_error,False,True
Variation,,
Control,25540,6475
Test,27034,9914


In [15]:
# H0: p_Error Test >= p_Error Control
# H1: p_Error Test < p_Error Control

#sl = 5%

count = np.array([9914, 6475])

nobs = np.array([(9914+27034), (6475+25540)])


proportions_ztest(count = count , nobs=nobs, alternative="smaller")
z_stat, p_value = proportions_ztest(count=count, nobs=nobs, alternative="smaller")

print("z-statistic:", z_stat)
print("p-value:", p_value)

z-statistic: 20.33058288094039
p-value: 1.0


### Hypothesis Testing: Error Rate

To evaluate whether the redesigned interface reduces user errors, we compared the error rates between the Test and Control groups.

**Hypotheses**

- **H₀:** p(Test) ≥ p(Control)  
- **H₁:** p(Test) < p(Control)

Significance level: α = 0.05

A one-sided two-proportion z-test was conducted.

**Test Result**

- z-statistic: 20.33  
- p-value: 1.00  

**Interpretation**

Since the p-value is much greater than 0.05, we fail to reject the null hypothesis.

This means there is no statistical evidence that the redesigned interface reduces the error rate compared to the existing interface.


## 3. Time spent on the Journey Test Analysis


To calculate the time spent during a user journey, we measure the time difference between consecutive steps within each visit.

For each `visit_id`, we compute the time difference between successive `date_time` values to estimate the time spent at each step.

Note: The dataframe is first sorted by `visit_id` and `date_time` to ensure that the steps are processed in chronological order.

In [19]:
df_web["date_time"] = pd.to_datetime(df_web["date_time"])

In [20]:
df_web["time_diff"] = df_web.groupby("visit_id")["date_time"].diff()
df_web["time_diff"] = df_web["time_diff"].dt.total_seconds()

In [25]:
df_web.groupby(["Variation"])["time_diff"].mean().reset_index()

,Variation,time_diff
0,Control,83.645064
1,Test,84.200540


In [27]:
# H0: mu Test = mu Control
# H1: mu Test != mu Control

#sl = 5%

test_s1 = df_web[(df_web["Variation"]=="Test") ]["time_diff"].dropna()
control_s1 = df_web[(df_web["Variation"]=="Control")]["time_diff"].dropna()

st.ttest_ind(test_s1, control_s1)
t_stat, p_value = st.ttest_ind(test_s1, control_s1)

print("t-statistic:", t_stat)
print("p-value:", p_value)

t-statistic: 0.6370550224975344
p-value: 0.5240895982049846



**Hypotheses**

- **H₀:** μ(Test) = μ(Control)  
- **H₁:** μ(Test) ≠ μ(Control)

Significance level: α = 0.05

An independent two-sample t-test is performed to determine whether there is a statistically significant difference in the average time spent between steps for the two groups.

Test result:

- t-statistic: 0.64  
- p-value: 0.52  

Since the p-value (0.52) is greater than the significance level (α = 0.05), we fail to reject the null hypothesis.

### Interpretation

There is no statistically significant difference in the average time spent between steps for users in the Test and Control groups.  

This suggests that while the redesigned interface improved the completion rate, it did not significantly change how long users spend navigating between steps in the process.